<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/geo4bolivia_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Find, load, understand and analyse Bolivia's subnational data with `geo4bolivia`

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

[`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) is the observatory's Python package. It is the way into the observatory's data: the SDG Atlas of Bolivia 2026 (the Sustainable Development Index, 15 goal indices and 81 indicators near 2012 and near 2024), population, night-time lights and satellite embeddings, for Bolivia's 343 municipalities and indigenous autonomies. It also runs the standard analyses of spatial data science on them. This notebook goes through the package in nine steps:

1. **Discover** what is available, what each variable means and how to cite it
2. **Load** a panel, with and without the package
3. **Understand** a variable: summaries and labels
4. **Compare the two periods**: the change of each unit
5. **Move across scales**: municipalities, provinces, departments and the country
6. **Link your own table** by name or INE code
7. **Build an indicator from a raster**
8. **Interpolate to a grid** and **save** your dataset in the observatory's format
9. **Analyse**: maps, spatial dependence, convergence, inequality and regression, one call each

Every function is documented, with an example, in the [package documentation](https://quarcs-lab.github.io/geo4bolivia/reference/).

## 0. Setup

In Google Colab the next cell installs the package (a few seconds: Colab already has most of its dependencies). The `[raster]` extra adds rasterio, which step 7 needs. Locally, `uv sync` has installed everything already.

In [1]:
import importlib.util

if importlib.util.find_spec("geo4bolivia") is None:
    !pip install -q "geo4bolivia[raster]"

In [2]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import plotly.express as px
import plotly.io as pio

import geo4bolivia as geo

if "google.colab" not in sys.modules:
    pio.renderers.default = "notebook_connected"   # keeps the saved notebook small
warnings.filterwarnings("ignore", module="numba")   # a notice of an optional library
print("geo4bolivia", geo.__version__)

geo4bolivia 0.6.0


## 1. Discover what is available

The catalog lists every dataset with its scale, its years and the number of units with data. The SDG Atlas has two periods for all 343 units; the datasets DS4Bolivia computed on the older 2015 map (population, lights, embeddings) cover the 333 municipalities whose territory has not changed since.

In [3]:
geo.datasets()[["dataset", "title", "years", "units", "variables"]]

,dataset,title,years,units,variables
0,pop,Population,2001–2020,333,2
1,ntl,Night-time lights per capita,2012–2020,333,2
2,sdg,"Sustainable Development Index, IDS (SDG Atlas ...","2012, 2024",343,4
3,sdg01,SDG 1: No poverty (SDG Atlas of Bolivia 2026),"2012, 2024",343,5
4,sdg02,SDG 2: Zero hunger (SDG Atlas of Bolivia 2026),"2012, 2024",343,8
5,sdg03,SDG 3: Good health and well-being (SDG Atlas o...,"2012, 2024",343,8
6,sdg04,SDG 4: Quality education (SDG Atlas of Bolivia...,"2012, 2024",343,12
7,sdg05,SDG 5: Gender equality (SDG Atlas of Bolivia 2...,"2012, 2024",343,7
8,sdg06,SDG 6: Clean water and sanitation (SDG Atlas o...,"2012, 2024",343,4
9,sdg07,SDG 7: Affordable and clean energy (SDG Atlas ...,"2012, 2024",343,5


`geo.search()` finds variables by words in their name, label or description, ignoring accents and case:

In [4]:
geo.search("poverty")[["dataset", "variable", "label", "years"]]

,dataset,variable,label,years
0,sdg01,index_sdg1,No poverty index (0-100),"2012, 2024"
1,sdg01,sdg1_2_mpi,Multidimensional poverty index (0-1),"2012, 2024"
2,sdg01,sdg1_4_eep,Energy poverty (% of dwellings),"2012, 2024"
3,sdg05,sdg5_1_pgm,Gender parity in multidimensional poverty (mal...,"2012, 2024"
4,sdg10,sdg10_2_pel,Parity of multidimensional poverty by ethnicit...,"2012, 2024"
5,sdg10,sdg10_2_mre,Parity of multidimensional poverty by migratio...,"2012, 2024"
6,sdg10,sdg10_2_tpm,Parity of multidimensional poverty for older p...,"2012, 2024"
7,sdg10,sdg10_2_gip,Gini coefficient of multidimensional poverty (...,"2012, 2024"
8,sdg05,index_sdg5,Gender equality index (0-100),"2012, 2024"
9,sdg10,index_sdg10,Reduced inequalities index (0-100),"2012, 2024"


`geo.info()` gathers a dataset's documentation: what it is, its coverage, the actual years behind its periods, its license, its sources and their licenses, the recommended citation and notes.

In [5]:
geo.info("sdg03")

SDG 3: Good health and well-being (SDG Atlas of Bolivia 2026)  [sdg03]

  The SDG 3 (Good health and well-being) index of the Atlas de los ODS en Bolivia 2026 (SDSN
  Bolivia) and the 7 indicators behind it, for Bolivia's 343 municipalities and indigenous
  autonomies (GAIOC), at two points in time: near the 2012 census and near the 2024 census,
  stored as 2012 and 2024 so that their change can be studied.

Coverage
  - 343 municipality units, 2012, 2024, 8 variables

Actual years behind the stored years (geo.reference_years)
  - 2012: ~2012; 2024: ~2024  (1 variables)
  - 2012: 2012; 2024: 2024  (2 variables)
  - 2012: 2008; 2024: 2023  (2 variables)
  - 2012: average 2011–2013; 2024: average 2022–2024  (3 variables)

License of the published files
  - Creative Commons Attribution 4.0 International License

Sources
  - Andersen, L. E., & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y
  tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La
  Paz: Universidad Privada Boliviana, SDSN Bolivia.
    License: No open license stated (© Universidad Privada Boliviana); used and shared with the
    permission of SDSN Bolivia

Cite as (geo.cite)
  GeoDevelopment Observatory of Bolivia (2026). SDG 3: Good health and well-being, from the
  Atlas de los ODS en Bolivia 2026 (343 municipalities, ~2012 and ~2024).
  https://github.com/quarcs-lab/gdo4bolivia/blob/main/docs/datasets/sdg03.md

Notes
  - Both periods are stored at nominal years: 2012 for the Atlas's initial values (~2012) and
  2024 for its final values (~2024). Each indicator's actual reference years differ (e.g. 2008
  and 2023 for the health survey, three-year averages for volatile indicators) and are given in
  its description and in `geo:reference_years`.
  - Load one variable in both periods, e.g. `geo.load("sdg03", variables="index_sdg3",
  names=True)`, and its change per unit with `geo.change(df, "index_sdg3")`.
  - The units are the 343 municipalities and indigenous autonomies (GAIOC) of 2025. Four of them
  were created after the 2012 census (TIOC-Raqaypampa, San Pedro de Macha, TIOC-Jatun Ayllu Yura
  and TIOC-Territorio Indígena Multiétnico); the Atlas gives them values in both periods.
  - Aggregating to provinces, departments or Bolivia (`geo.aggregate`, `geo.combine`) averages
  the municipal values weighted by census population (`weighted_mean:population`; the 2012
  census for ~2012 and the 2024 census for ~2024, loaded from the `sdg` dataset through
  `geo:weights`): e.g. `geo.aggregate(geo.load("sdg03"), to="department", dataset="sdg03")`.
  These are population-weighted approximations, not the Atlas's own department figures.

Notebooks
  - Find, load and understand Bolivia's subnational data with geo4bolivia
  - Dynamics of sustainable development in Bolivia, 2012–2024
  - Poverty in Bolivia: an extended EDA and ESDA

Documentation: https://github.com/quarcs-lab/gdo4bolivia/blob/main/docs/datasets/sdg03.md

`geo.dictionary()` describes every column: readable label, definition, unit, aggregation rule and, when they differ from the stored years, the actual reference years.

In [6]:
cols = ["variable", "label", "unit", "aggregation", "reference_years"]
geo.dictionary("sdg03")[cols]

,variable,label,unit,aggregation,reference_years
0,geo_id,Municipality ID,,,
1,year,Period,,,
2,mun_id,INE municipality code,,,
3,index_sdg3,Good health and well-being index (0-100),index (0-100),weighted_mean:population,2012: ~2012; 2024: ~2024
4,sdg3_1_cpp,Births attended by skilled health personnel (%),% of births,weighted_mean:population,2012: 2012; 2024: 2024
5,sdg3_2_tmi,"Infant mortality (per 1,000 live births)","deaths under 1 per 1,000 live births",weighted_mean:population,2012: 2008; 2024: 2023
6,sdg3_2_tmn,"Under-5 mortality (per 1,000 live births)","deaths under 5 per 1,000 live births",weighted_mean:population,2012: 2008; 2024: 2023
7,sdg3_3_vih,HIV incidence (per million people),cases per million people,weighted_mean:population,2012: average 2011–2013; 2024: average 2022–2024
8,sdg3_3_tub,"Tuberculosis incidence (per 100,000 people)","cases per 100,000 people",weighted_mean:population,2012: average 2011–2013; 2024: average 2022–2024
9,sdg3_3_vec,Vector-borne disease index (0-1),index (0-1),weighted_mean:population,2012: average 2011–2013; 2024: average 2022–2024


The Atlas stores its initial values at **2012** and its final values at **2024**, the years of the last two censuses, but some indicators come from other years: infant mortality, for example, from the health surveys of 2008 and 2023.

In [7]:
geo.reference_years("sdg03", "sdg3_2_tmi")

,variable,label,year,reference
0,sdg3_2_tmi,"Infant mortality (per 1,000 live births)",2012,2008
1,sdg3_2_tmi,"Infant mortality (per 1,000 live births)",2024,2023


If you use the data, cite the dataset and its original sources:

In [8]:
print(geo.cite("sdg03"))

GeoDevelopment Observatory of Bolivia (2026). SDG 3: Good health and well-being, from the Atlas de los ODS en Bolivia 2026 (343 municipalities, ~2012 and ~2024). https://github.com/quarcs-lab/gdo4bolivia/blob/main/docs/datasets/sdg03.md

Please also credit the original sources:
- Please cite the Atlas and its data as: Andersen, L.E. & Argandoña, F. (2026). Atlas de los ODS en Bolivia 2026: Brechas y tendencias del desarrollo sostenible en las Entidades Territoriales Autónomas, 2012–2024. La Paz: Universidad Privada Boliviana, SDSN Bolivia.


## 2. Load a panel, with and without the package

### Without the package

Every published file is a plain URL, so pandas, geopandas, R, Stata or Excel can read it directly:

- **`data/<scale>/<dataset>.csv`**: the indicators, as a panel with one row per unit (`geo_id`) and year
- **`geo/units.csv`**: names, parent units and the codes other sources use (INE, DS4Bolivia, GADM)
- **`geo/boundaries/<scale>.geojson`**: polygons, keyed by the same `geo_id`

In [9]:
BASE = "https://raw.githubusercontent.com/quarcs-lab/gdo4bolivia/main/"

# Inside a clone of the repository, read the local files instead (Colab uses URLs)
here = [Path.cwd(), *Path.cwd().parents]
local = next((p for p in here if (p / "geo" / "units.csv").is_file()), None)
if local is not None:
    BASE = local.as_posix() + "/"

sdg01 = pd.read_csv(BASE + "data/municipality/sdg01.csv")       # 343 units x 2 periods
units = pd.read_csv(BASE + "geo/units.csv", dtype={"ine_code": str})    # every scale
municipalities = gpd.read_file(BASE + "geo/boundaries/municipality.geojson")
print(sdg01.shape, units.shape, municipalities.shape)
sdg01.head()

(686, 8) (466, 15) (343, 3)


,geo_id,year,mun_id,index_sdg1,sdg1_2_mpi,sdg1_4_eep,sdg1_4_nbi,sdg1_4_ssb
0,BO-01-01-01,2012,10201,11.7,0.512,87.2,87.7,12.8
1,BO-01-01-01,2024,10201,35.7,0.187,82.0,71.7,29.6
2,BO-01-01-02,2012,10202,14.0,0.452,89.9,86.9,13.1
3,BO-01-01-02,2024,10202,38.9,0.133,84.6,69.7,33.5
4,BO-01-02-01,2012,10801,38.9,0.267,70.3,67.1,40.4


### With the package

`geo.load()` does the same in one line, and more: it picks variables and years, adds the unit names and the department or province each unit lies in, attaches boundaries, or reshapes to one column per year. Here is the Multidimensional Poverty Index (`sdg1_2_mpi`, 0 to 1, higher = poorer), which the Atlas computes from the censuses themselves:

In [10]:
mpi = geo.load("sdg01", variables="sdg1_2_mpi", groups=["department"])
mpi.head()

,geo_id,name,year,department,sdg1_2_mpi
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512
1,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187
2,BO-01-01-02,Tarvita,2012,Chuquisaca,0.452
3,BO-01-01-02,Tarvita,2024,Chuquisaca,0.133
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,0.267


In [11]:
ids_2024 = geo.load("sdg", years=2024, geometry=True)       # a GeoDataFrame
pop_wide = geo.load("pop", variables="pop", wide="pop")      # one column per year
print(type(ids_2024).__name__, ids_2024.shape, "|", pop_wide.shape)

GeoDataFrame (343, 9) | (333, 21)


`geo.units()` has every unit's name, parents and source codes, and `geo.boundaries()` its polygons:

In [12]:
cols = ["geo_id", "name", "ine_code", "province_id", "department_id", "area_km2"]
geo.units("municipality")[cols].head()

,geo_id,name,ine_code,province_id,department_id,area_km2
0,BO-01-01-01,Azurduy,010201,BO-01-01,BO-01,1729.28
1,BO-01-01-02,Tarvita,010202,BO-01-01,BO-01,1354.24
2,BO-01-02-01,Villa Serrano,010801,BO-01-02,BO-01,1730.36
3,BO-01-03-01,Huacareta,010502,BO-01-03,BO-01,2963.81
4,BO-01-03-02,Monteagudo,010501,BO-01-03,BO-01,3379.79


## 3. Understand a variable

`geo.describe()` summarises variables per period, with their labels and units; `by=` summarises each group instead.

In [13]:
edu = geo.load("sdg04", variables=["index_sdg4", "sdg4_6_alf"],
               groups=["department"])
geo.describe(edu)

,variable,label,unit,year,count,mean,std,min,p25,median,p75,max
0,index_sdg4,Quality education index (0-100),index (0-100),2012,343,47.559767,10.575112,18.6,42.15,47.6,53.35,84.8
1,index_sdg4,Quality education index (0-100),index (0-100),2024,343,69.047522,8.753478,43.5,64.00,69.5,74.35,94.7
2,sdg4_6_alf,"Literacy rate, 15+ (%)",% of people 15+,2012,343,91.731195,5.878491,68.3,88.65,93.7,95.90,100.0
3,sdg4_6_alf,"Literacy rate, 15+ (%)",% of people 15+,2024,343,93.355102,4.928841,73.0,91.35,94.9,96.80,99.5


In [14]:
geo.describe(edu[edu["year"] == 2024], variables="index_sdg4", by="department")

,variable,label,unit,department,count,mean,std,min,p25,median,p75,max
0,index_sdg4,Quality education index (0-100),index (0-100),Beni,20,67.420000,6.598373,56.3,64.300,67.25,70.90,84.2
1,index_sdg4,Quality education index (0-100),index (0-100),Chuquisaca,29,65.782759,9.110711,52.2,56.900,66.30,71.70,91.4
2,index_sdg4,Quality education index (0-100),index (0-100),Cochabamba,48,68.077083,12.562867,43.5,59.550,68.25,77.50,93.2
3,index_sdg4,Quality education index (0-100),index (0-100),La Paz,87,70.286207,5.975321,57.7,66.500,69.90,72.80,94.7
4,index_sdg4,Quality education index (0-100),index (0-100),Oruro,35,74.065714,6.115599,60.3,70.900,73.90,77.40,92.6
5,index_sdg4,Quality education index (0-100),index (0-100),Pando,15,66.446667,6.974634,56.7,63.550,65.30,68.20,82.1
6,index_sdg4,Quality education index (0-100),index (0-100),Potosí,42,66.200000,11.345570,46.3,56.575,65.90,75.55,89.7
7,index_sdg4,Quality education index (0-100),index (0-100),Santa Cruz,56,69.157143,7.052346,56.1,64.200,69.65,73.25,87.1
8,index_sdg4,Quality education index (0-100),index (0-100),Tarija,11,72.945455,7.534901,62.4,68.700,73.10,77.10,89.5


`geo.labels()` returns the label of every column, ready for the `labels=` argument of a Plotly Express figure of your own:

In [15]:
px.box(edu, x="department", y="sdg4_6_alf", color="year", labels=geo.labels(edu),
       title="Literacy by department, ~2012 and ~2024")

## 4. Compare the two periods

`geo.change()` turns a panel into one row per unit: its first and last values, the absolute, percentage and annual change, and its rank in each period (1 = highest).

In [16]:
ids = geo.load("sdg", variables="index_sdg", groups=["department"])
chg = geo.change(ids, "index_sdg")
cols = ["name", "department", "index_sdg_2012", "index_sdg_2024",
        "index_sdg_change", "index_sdg_rank_2012", "index_sdg_rank_2024"]
chg.nlargest(5, "index_sdg_change")[cols]

,name,department,index_sdg_2012,index_sdg_2024,index_sdg_change,index_sdg_rank_2012,index_sdg_rank_2024
194,Cruz de Machacamarca,Oruro,42.9,63.4,20.5,315,90
199,Carangas,Oruro,43.4,62.8,19.4,304,100
42,Sicaya,Cochabamba,42.8,60.4,17.6,318,153
67,Villa Gualberto Villarroel,Cochabamba,43.4,60.3,16.9,304,156
201,Todos Santos,Oruro,44.4,61.2,16.8,290,135


In [17]:
print(f"The IDS rose in {(chg['index_sdg_change'] > 0).sum()} of {len(chg)} units, "
      f"by {chg['index_sdg_change'].mean():.1f} points on average.")

The IDS rose in 342 of 343 units, by 10.0 points on average.


The five largest gains are in municipalities that started near the bottom of the ranking (ranks 290 to 318 of 343). Whether the units that started lower gained more in general is a question for step 9.

## 5. Move across scales

Every geo ID nests by prefix: municipality `BO-04-17-04` lies in province `BO-04-17`, in department `BO-04`, in Bolivia `BO`. `geo.aggregate()` moves a panel up with each variable's rule from the metadata. The SDG indicators average with census-population weights (the 2012 census for ~2012, the 2024 census for ~2024), which the package loads from the `sdg` dataset by itself. These are approximations, not the Atlas's own department figures. `n_units` and `coverage` tell how many municipalities each result rests on.

In [18]:
up = geo.aggregate(mpi, to=["department", "country"], dataset="sdg01")
up.tail(4)

,geo_id,scale,year,sdg1_2_mpi,n_units,coverage
16,BO-09,department,2012,0.079347,11,1.0
17,BO-09,department,2024,0.017672,11,1.0
18,BO,country,2012,0.133185,343,1.0
19,BO,country,2024,0.033046,343,1.0


In [19]:
dep = geo.add_names(up[up["scale"] == "department"])
px.line(dep, x="year", y="sdg1_2_mpi", color="name", markers=True,
        labels=geo.labels(dep),
        title="Multidimensional poverty by department (census-population weighted)")

Night-time lights cover 333 of the 343 municipalities, and the coverage shows it. Their logs have no aggregation rule, so an explicit rule is needed:

In [20]:
ntl_dep = geo.aggregate(geo.load("ntl", years=2020), to="department",
                        rules={"ln_NTLpc": "mean"}, variables="ln_NTLpc")
ntl_dep[["geo_id", "ln_NTLpc", "n_units", "coverage"]]

,geo_id,ln_NTLpc,n_units,coverage
0,BO-01,13.782270,29,1.000000
1,BO-02,14.479835,46,0.958333
2,BO-03,13.634814,15,0.750000
3,BO-04,13.359547,87,1.000000
4,BO-05,14.081636,35,1.000000
5,BO-06,12.719444,13,0.866667
6,BO-07,13.658344,36,0.857143
7,BO-08,14.555343,56,1.000000
8,BO-09,14.986451,11,1.000000


`geo.combine()` puts several datasets on one scale, aggregating them as needed:

In [21]:
goals = geo.combine(["sdg03", "sdg04"], scale="department", years=2024)
goals = geo.add_names(goals)
px.scatter(goals, x="index_sdg3", y="index_sdg4", text="name",
           labels=geo.labels(goals), title="Health and education by department, ~2024")

## 6. Link your own table

Tables from reports and ministries identify units by name, spelled in many ways, or by INE code. `geo.link()` attaches the observatory's `geo_id`. It ignores accents, case, spaces and punctuation, and tries every name and code column in `geo.units()`.

The values below are **made up** for the demonstration; replace them with your own data.

In [22]:
mine = pd.DataFrame({
    "Departamento": ["LA PAZ", "Cochabamba", "Santa Cruz", "Potosi", "Chuquisaca",
                     "La Paz"],
    "year":         [2024, 2024, 2024, 2024, 2024, 2025],
    "clinics":      [412, 301, 388, 266, 190, 420],          # illustrative values
})
mine = geo.link(mine, scale="department", on="Departamento")
print(geo.check(mine, scale="department"))   # [] = a valid panel
mine

[]


,geo_id,Departamento,year,clinics
0,BO-04,LA PAZ,2024,412
1,BO-02,Cochabamba,2024,301
2,BO-08,Santa Cruz,2024,388
3,BO-07,Potosi,2024,266
4,BO-01,Chuquisaca,2024,190
5,BO-04,La Paz,2025,420


Municipal statistics from INE (and every DS4Bolivia table) carry codes. Link them by code: INE's 6-digit `ine_code`, the same code as a number (`mun_id`), or DS4Bolivia's `asdf_id`. The names follow the Atlas, so the capital is *Nuestra Señora de La Paz*:

In [23]:
ine = pd.DataFrame({"codigo": [20101, 20105, 70101, 30101, 10101],
                    "teachers": [9120, 6211, 11034, 5980, 2870]})   # illustrative
linked = geo.link(ine, scale="municipality", on="codigo", key="mun_id")
geo.add_names(linked, groups=["department"])

,geo_id,name,department,codigo,teachers
0,BO-04-17-04,Nuestra Señora de La Paz,La Paz,20101,9120
1,BO-04-17-02,El Alto,La Paz,20105,6211
2,BO-08-01-05,Santa Cruz de la Sierra,Santa Cruz,70101,11034
3,BO-02-08-01,Cochabamba,Cochabamba,30101,5980
4,BO-01-06-02,Sucre,Chuquisaca,10101,2870


Older spellings still link: by default `geo.link()` also tries `name_ds4bolivia` and `name_gadm`, so "La Paz" finds the capital. A name shared by several municipalities, or a typo, stops with an explanation instead of silently linking the wrong unit:

In [24]:
for value in ["San Javier", "Cochabmba"]:
    table = pd.DataFrame({"m": [value]})
    try:
        geo.link(table, scale="municipality", on="m", key="name")
    except ValueError as err:
        print(err, "\n")

1 row(s) of 'm' name several municipality units and were not linked: 'San Javier' (BO-03-01-01, BO-08-10-04). Link them by a code (key=...) instead. 

1 value(s) of 'm' match no municipality 'name': 'Cochabmba' (did you mean 'Cochabamba'?). Fix them, e.g. df['m'] = df['m'].replace({...}), or pass errors='warn'. 



## 7. Build an indicator from a raster

`geo.zonal()` summarises any GeoTIFF (for example an export from Google Earth Engine, or a WorldPop population grid) over the units of a scale. As a self-contained example, we compute a raster of **distance to the nearest of Bolivia's three largest cities** (La Paz, Cochabamba and Santa Cruz de la Sierra, the "central axis"), a simple proxy of market access, and average it over municipalities.

In [25]:
import rasterio
from rasterio.transform import from_bounds

CITIES = {"La Paz": (-16.4955, -68.1336), "Cochabamba": (-17.3895, -66.1568),
          "Santa Cruz": (-17.7833, -63.1821)}

x0, y0, x1, y1 = geo.boundaries("country").total_bounds
nx, ny = 900, 1000                                             # about 1.5 km pixels
LON, LAT = np.meshgrid(np.linspace(x0, x1, nx), np.linspace(y1, y0, ny))

def haversine_km(lat, lon, lat0, lon0):
    lat, lon, lat0, lon0 = map(np.radians, (lat, lon, lat0, lon0))
    a = (np.sin((lat - lat0) / 2) ** 2
         + np.cos(lat0) * np.cos(lat) * np.sin((lon - lon0) / 2) ** 2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))

km = np.min([haversine_km(LAT, LON, la, lo) for la, lo in CITIES.values()], axis=0)

with rasterio.open("distance_to_main_cities.tif", "w", driver="GTiff", width=nx,
                   height=ny, count=1, dtype="float32", crs="EPSG:4326",
                   transform=from_bounds(x0, y0, x1, y1, nx, ny)) as dst:
    dst.write(km.astype("float32"), 1)

access = geo.zonal("distance_to_main_cities.tif", scale="municipality",
                   stats=["mean", "max"], name="dist_axis_km")
access.head()

,geo_id,dist_axis_km_mean,dist_axis_km_max
0,BO-01-01-01,297.976007,328.039978
1,BO-01-01-02,265.125369,289.335236
2,BO-01-02-01,176.702818,197.327393
3,BO-01-03-01,325.066672,367.762909
4,BO-01-03-02,265.958876,325.305908


The result has one row per municipality and a `geo_id`, so it joins any dataset of the observatory. Step 9 uses it in a regression.

## 8. Interpolate to a grid and save

Grids, watersheds or a different municipality map do not nest in the administrative hierarchy. `geo.overlay()` moves data onto them by area weights. Totals (`extensive`) are split by area; rates (`intensive`) are area-weighted means. Here the census population of 2024 goes onto a 50 km grid, assuming that people are spread evenly within each municipality, and `geo.map()` draws it: it maps any table with a `geo_id` column and a geometry.

In [26]:
from shapely.geometry import box

crs = geo.config()["equal_area_crs"]          # Albers equal-area for Bolivia, metres
country = geo.boundaries("country").to_crs(crs)
gx0, gy0, gx1, gy1 = country.total_bounds
size = 50_000
cells = [box(x, y, x + size, y + size)
         for x in np.arange(gx0, gx1, size) for y in np.arange(gy0, gy1, size)]
grid = gpd.GeoDataFrame({"geo_id": [f"c{i}" for i in range(len(cells))]},
                        geometry=cells, crs=crs)
grid = grid[grid.intersects(country.geometry.iloc[0])].to_crs("EPSG:4326")

census_2024 = geo.load("sdg", variables="population", years=2024)
on_grid = geo.overlay(census_2024, grid, extensive=["population"], id_col="geo_id")
print(f"municipalities: {census_2024['population'].sum():,.0f}  "
      f"grid: {on_grid['population'].sum():,.0f}")              # the total is kept

geo.map(grid.merge(on_grid, on="geo_id"), "population", scheme="quantiles",
        title="Census population 2024 on a 50 km grid")

municipalities: 11,364,933  grid: 11,364,933


Finally, save your indicator exactly like the published datasets: CSV, Parquet and a Frictionless datapackage describing every variable. It is then easy to share, reload or contribute to the observatory ([CONTRIBUTING_DATA.md](https://github.com/quarcs-lab/gdo4bolivia/blob/main/CONTRIBUTING_DATA.md)).

In [27]:
access["year"] = 2024
variables = {
    "dist_axis_km_mean": {"label": "Mean distance to the central axis (km)",
                          "description": "Mean distance to the nearest main city",
                          "unit": "km", "aggregation": "none"},
    "dist_axis_km_max": {"label": "Maximum distance to the central axis (km)",
                         "description": "Maximum distance to the nearest main city",
                         "unit": "km", "aggregation": "max"},
}
datapackage = geo.save(
    access, "market_access", scale="municipality", directory="my_data",
    title="Distance to the central axis", variables=variables,
    description="Mean and maximum straight-line distance from each municipality to "
                "the nearest of La Paz, Cochabamba and Santa Cruz de la Sierra.")

wrote my_data/data/municipality/market_access.csv, my_data/data/municipality/market_access.parquet and my_data/metadata/market_access.datapackage.json


## 9. Analyse

The package also runs the standard analyses of spatial data science. Each one is a single call that takes a panel (or just a dataset id), finds the boundaries and builds the spatial weights by itself. This step shows one call per family on one indicator, the Sustainable Development Index (IDS, 0 to 100) of the dataset `sdg`, joined with the distance indicator of step 7. The observatory's variables take their labels from the metadata; `geo.labels(..., own=...)` gives a label to a column of your own, which the figures, tables and readings then use.

In [28]:
panel = geo.load("sdg", groups=["department"]).merge(
    access[["geo_id", "dist_axis_km_mean"]], on="geo_id")
geo.labels(panel, own={"dist_axis_km_mean": "Mean distance to the central axis (km)"})
panel.head(4)

,geo_id,name,year,department,mun_id,index_sdg,rank_sdg,population,urbanization,dist_axis_km_mean
0,BO-01-01-01,Azurduy,2012,Chuquisaca,10201,40.8,332,10652,NaN,297.976007
1,BO-01-01-01,Azurduy,2024,Chuquisaca,10201,54.5,297,7903,0.0,297.976007
2,BO-01-01-02,Tarvita,2012,Chuquisaca,10202,44.5,287,14261,NaN,265.125369
3,BO-01-01-02,Tarvita,2024,Chuquisaca,10202,53.9,306,16812,0.0,265.125369


### Maps

`geo.map()` draws any variable. With a classification scheme and a list of years, it puts the periods side by side in the same classes (here natural breaks computed over both years), so a colour means the same on both maps. `change=True` maps the change instead.

In [29]:
geo.map(panel, "index_sdg", year=[2012, 2024], scheme="fisher_jenks")

### Spatial dependence, and the result object

Is sustainable development clustered in space? Global Moran's I answers for the whole country.

In [30]:
res = geo.moran(panel, "index_sdg", year=2024)
res

Global Moran's I for **Sustainable Development Index, IDS (0-100)** in 2024 is 0.468, against an expectation of -0.00292 under spatial randomness — statistically significant at the 1% level (pseudo p = 0.001 from 999 permutations, under queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343).

The dependence is **positive**: similar values cluster in space — high values sit next to high values and low next to low, so the map shows contiguous patches rather than a random scatter.

261 of 343 units (76%) fall in the clustering quadrants of the scatter (High-High or Low-Low); the rest are surrounded by neighbors unlike themselves. The slope of the fitted line equals Moran's I under row-standardized weights, so a steeper line means stronger clustering.

_These are associations, not causal effects. A causal reading needs a research design._

Every analysis returns a **result object**. Displayed, as above, it shows its main figure and a reading of the numbers. Its parts are there to use one by one:

| Attribute | What it holds |
| --- | --- |
| `res.fig` | the main figure (Plotly, interactive); some results have more, e.g. `fig_map` |
| `res.df` | the table behind it, one row per unit or year, with `geo_id` and `name` |
| `res.gt` | a formatted table, where one helps (regressions, summaries) |
| `res.moran_i`, `res.p_sim`, ... | the numbers, by name |
| `res.interpret()` | the reading in words; it describes associations, never causes |
| `res.notes` | what the function decided for you (units left out, years used) |

In [31]:
print(round(res.moran_i, 3), res.p_sim, "|", res.w_spec)
res.df.head(3)

0.468 0.001 | queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343


,geo_id,name,value,lag,quadrant
0,BO-01-01-01,Azurduy,-1.061887,-0.376963,LL
1,BO-01-01-02,Tarvita,-1.176040,-0.649664,LL
2,BO-01-02-01,Villa Serrano,0.726525,0.853362,HH


*Where* are the clusters? Local Moran statistics (LISA) put them on the map:

In [32]:
geo.lisa(panel, "index_sdg", year=2024)

Local Moran statistics (LISA) locate *where* **Sustainable Development Index, IDS (0-100)** in 2024 clusters or stands out, under queen contiguity (1 island(s) attached to nearest neighbor), row-standardized, n=343. The accompanying global Moran's I is 0.468 (pseudo p = 0.001), consistent with overall clustering of similar values.

At the 0.05 significance level, 116 of 343 units show significant local association: **60 High-High** hot spots (high values surrounded by high neighbors) and **45 Low-Low** cold spots (low surrounded by low) mark clustering, while **6 High-Low** and **5 Low-High** units are spatial outliers that break with their surroundings. The remaining 227 units are not significant — their local pattern is compatible with randomness.

LISA pseudo p-values are computed unit by unit without a multiple-testing adjustment, so treat borderline clusters cautiously and read the map as descriptive of where dependence concentrates.

_These are associations, not causal effects. A causal reading needs a research design._

### Convergence

Did the municipalities that started lower catch up? β-convergence regresses each unit's yearly growth between the two periods on the logarithm of its initial level; a negative slope means that the lower a unit started, the faster it grew.

In [33]:
geo.beta_convergence(panel, "index_sdg")

Across 343 units, the growth of **Sustainable Development Index, IDS (0-100)** over a 12-year window is associated with its initial log level with a total slope of **-0.0287** (SE 0.00171), statistically significant at the 1% level.
The slope is negative — the β-convergence pattern: units that started lower tended to grow faster, narrowing initial gaps.
That slope implies a convergence speed of λ = 0.0351 per year and a half-life of about 19.7 years — the time for half of an initial gap to close at this pace.

_These are associations, not causal effects. A causal reading needs a research design._

### Inequality

Did the gaps between municipalities narrow? `geo.inequality()` computes the Gini and Theil indices in each year; `.fig_lorenz` has the Lorenz curves.

In [34]:
geo.inequality(panel, "index_sdg")

Cross-sectional inequality in **Sustainable Development Index, IDS (0-100)** is tracked across 343 units over 2 years (2012 to 2024).

The Gini index fell from 0.065 to 0.0493 between the first and last year.

The Theil index fell from 0.00663 to 0.00381 between the first and last year.

With two years (2012 and 2024) the measures can be compared, but their trend cannot be tested: that needs three or more years.

On the Lorenz curve of 2024, the half of the units with the lowest values holds 46% of the total (45% in 2012) and the tenth with the highest values holds 12% (12%); with no inequality they would hold 50% and 10%.

_These are associations, not causal effects. A causal reading needs a research design._

### Regression

What goes with a high index? `geo.regression()` estimates several specifications side by side. Here the first has the urbanization rate alone; the second adds the distance to the central axis built in step 7 and compares municipalities within the same department (`fixed_effects`). `.gt` is the regression table.

In [35]:
reg = geo.regression(panel, "index_sdg",
                     [["urbanization"], ["urbanization", "dist_axis_km_mean"]],
                     fixed_effects=[None, "department"])
reg.gt

GT(_tbl_data=                                     term        (1)       (2)
0     Urbanization rate (% of population)   0.094***  0.098***
1                                            (0.008)   (0.008)
2  Mean distance to the central axis (km)               -0.002
3                                                      (0.003)
4                               Intercept  57.582***          
5                                            (0.313)          
6                Department fixed effects         No       Yes
7                            Observations        343       343
8                                      R²      0.313     0.430
9                             Adjusted R²      0.311     0.413, _body=<great_tables._gt_data.Body object at 0x1378f1850>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None, stub_level=None), ColInfo(var='(1)', type=<ColInfoTypeEnum.default: 1>, column_label='(1)', column_align='center', column_width=None, stub_level=None), ColInfo(var='(2)', type=<ColInfoTypeEnum.default: 1>, column_label='(2)', column_align='center', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x137913110>, _spanners=Spanners([SpannerInfo(spanner_id='Sustainable Development Index, IDS (0-100)', spanner_level=0, spanner_label='Sustainable Development Index, IDS (0-100)', spanner_units=None, spanner_pattern=None, vars=['(1)', '(2)'], built=None)]), _heading=Heading(title='Regression of Sustainable Development Index, IDS (0-100)', subtitle='Ordinary least squares, 2024', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1378f13d0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1378f1210>, _source_notes=['*** p<0.01, ** p<0.05, * p<0.10. Standard errors in parentheses: heteroskedasticity-robust (HC1).'], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=[6], mask=None), grpname=None, colname='term', rownum=6, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')]), StyleInfo(locname=LocBody(columns=None, rows=[6], mask=None), grpname=None, colname='(1)', rownum=6, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')]), StyleInfo(locname=LocBody(columns=None, rows=[6], mask=None), grpname=None, colname='(2)', rownum=6, colnum=None, styles=[CellStyleBorders(sides='top', color='#7f7f7f', style='solid', weight='1px')])], _locale=<great_tables._gt_data.Locale object at 0x1378f1410>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=T

In [36]:
reg

**Ordinary least squares** of **Sustainable Development Index, IDS (0-100)**, 2024: 2 specifications side by side. Model (2), the last one, reads as follows.

- **Urbanization rate (% of population)**: a one-unit higher value goes with Sustainable Development Index, IDS (0-100) that is 0.0977 higher (statistically significant at the 1% level; 95% interval 0.0819 to 0.113), the other regressors held constant, within the same department.
- **Mean distance to the central axis (km)**: a one-unit higher value goes with Sustainable Development Index, IDS (0-100) that is 0.00231 lower (not statistically significant at conventional levels; 95% interval -0.00767 to 0.00306), the other regressors held constant, within the same department.

Fixed effects for *Department* absorb every difference between their groups, so the coefficients compare observations of the same group only.
The model has 343 observations and R² = 0.43 (adjusted 0.413): the regressors and the fixed effects account for 43.0% of the variation in Sustainable Development Index, IDS (0-100).
Standard errors are heteroskedasticity-robust (HC1).

Across the models, the coefficient on **Urbanization rate (% of population)** is (1) 0.094, (2) 0.0977. It is stable: a coefficient that moves when regressors or fixed effects are added was partly picking up what they measure; one that stays put does not depend on them.

_These are associations, not causal effects. A causal reading needs a research design._

Together, the five results say this. Sustainable development is clustered in space (Moran's I of 0.47 in 2024): the LISA map finds 60 municipalities in high-value clusters and 45 in low-value clusters. The municipalities that started lower grew faster (a slope of -0.029, which closes half of a gap in about 20 years), and the Gini index fell from 0.065 to 0.049. In 2024, ten more percentage points of urban population go with an index about one point higher, also among municipalities of the same department, while the distance to the central axis shows no association with the index once urbanization and the department are taken into account (-0.002 points per km, not statistically significant). These are associations, not causal effects.

### The rest of the package

The same pattern, a call that returns a result object, covers the other analyses:

| Family | Functions |
| --- | --- |
| Explore | `correlation`, `extremes`, `distribution`, `scatter`, `trend` |
| Maps and spatial dependence | `map`, `weights`, `connectivity`, `moran`, `lisa` |
| Convergence | `beta_convergence`, `sigma_convergence`, `convergence_clubs` |
| Transitions and mobility | `transitions`, `spatial_transitions`, `lisa_transitions`, `rank_mobility` |
| Inequality | `inequality`, `theil_decomposition` |
| Models | `regression`, `panel_model`, `spatial_model`, `spatial_diagnostics` |
| Spatial heterogeneity | `mgwr` |

A few of them need an extra library, installed with the package when you ask for it: `"geo4bolivia[dynamics]"` (`transitions`, `spatial_transitions`), `"geo4bolivia[panel]"` (`panel_model`), `"geo4bolivia[mgwr]"` (`mgwr`), or `"geo4bolivia[all]"`. Your own choices replace the defaults: `geo.weights()` builds other spatial weights to pass as `w=`, and `gdf=` supplies boundaries for units the package does not know.

## Next steps

- The [package documentation](https://quarcs-lab.github.io/geo4bolivia/): [Get started](https://quarcs-lab.github.io/geo4bolivia/get-started.html), [Explore](https://quarcs-lab.github.io/geo4bolivia/explore.html), [Analyze](https://quarcs-lab.github.io/geo4bolivia/analyze.html) and the [function reference](https://quarcs-lab.github.io/geo4bolivia/reference/), with an example per function
- One question, did Bolivia's municipalities converge, answered with geo4bolivia and its companion packages expdpy and geometrics: [`three_packages.ipynb`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/three_packages.ipynb)
- Exploring multidimensional poverty with the exploration functions: [`eda.ipynb`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/eda.ipynb)
- Change, convergence and rank mobility of the 15 goals: [`sdg_dynamics.ipynb`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/sdg_dynamics.ipynb)
- How the data are organised: [the data infrastructure](https://github.com/quarcs-lab/gdo4bolivia/blob/main/docs/data-infrastructure.md)
- Every function also has help: `help(geo.change)`